# Do Players Notice the AI, or Just the Quality?
**Author:** Neha Kollikara and Tomas Aguilar

**Course:** INFO 4340 - Analytics and AI

**Question:** Among buyers who dislike AI-disclosed Steam games, how often is generative AI the stated reason, compared with quality? (Q2, exploratory: when reviewers don't mention AI, do complaints concentrate where AI was used?)

**Notebook Parts:**
1. Pilot: keyword search and hand check (development set)
2. Clean: raw scrape to `data/reviews_clean.csv`
3. Keyword baseline *(coming)*
4. Claude classifier *(coming)*
5. Evaluation *(coming)*
6. Analysis *(coming)*

# Setup

In [32]:
%pip install -q openpyxl


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [58]:
#Imports
import pandas as pd
import numpy as np
import os
from scipy import stats
from sklearn.metrics import cohen_kappa_score
import json
import time
import anthropic
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from sklearn.metrics import classification_report, confusion_matrix

# Part 1: Pilot
**Purpose:** Before the full scrape, estimate how often negative reviews of AI-disclosed games mention generative AI, and test a keyword search.

**Data:** 312 English negative reviews (Jan 2024 – Dec 2025) from the 5 lowest-rated AI-disclosed games at the time, scraped Nov 4, 2026.

**Outputs:** `data/pilot.csv` (development set, excluded from all testing) and `data/pilot_hits_labeled.csv` (hand-checked keyword hits).

**Key results:** 8 of 15 keyword hits (53%) were about generative AI; estimated rate 2.6% (95% CI 1.3–5.0%).

## Pull the Pilot

In [3]:
# Pull the pilot (already run on Nov 4, 2026; left commented out so re-running the notebook doesn't change the data)

# %run steam_reviews_scraper.py --appids 2738870 3484300 2687970 2943670 3444080 --max-per-game 100 --out data/pilot.csv

## Load and Check

In [4]:
# Load the pilot and confirm it matches what was scraped
pilot = pd.read_csv("data/pilot.csv")
print(f"Loaded: {pilot.shape[0]} rows, {pilot.shape[1]} columns")
print(pilot["appid"].value_counts())

assert len(pilot) == 312, f"Expected 312 pilot reviews, found {len(pilot)}"
assert pilot["appid"].nunique() == 5, "Pilot should cover 5 games"
assert (pilot["voted_up"] == False).all(), "Pilot should contain only negative reviews"
assert pilot["recommendationid"].is_unique, "Review IDs should be unique"

Loaded: 312 rows, 18 columns
appid
3484300    100
2687970    100
2738870     48
2943670     41
3444080     23
Name: count, dtype: int64


## Keyword Search

In [5]:
# Keyword search for AI mentions
# This pattern is fixed here and reused unchanged as the keyword baseline, so it isn't tuned on test data
AI_PATTERN = r"\bai\b|a\.i\.|ai-generated|generative|genai|slop|midjourney|chatgpt|artificial intelligence"

hits = pilot[pilot["review"].str.contains(AI_PATTERN, case=False, regex=True)].copy()
print(f"{len(hits)} of {len(pilot)} reviews match the keyword pattern ({len(hits)/len(pilot):.1%})")
print(hits.groupby("appid").size())

15 of 312 reviews match the keyword pattern (4.8%)
appid
2687970     3
2943670     1
3484300    11
dtype: int64


## Export for Hand-Check

In [6]:
# Export keyword hits for hand labeling
# Only writes the file if it doesn't exist, so re-running never erases the hand labels
out_file = "data/pilot_hits_labeled.csv"
if not os.path.exists(out_file):
    hits["label"] = ""
    hits["note"] = ""
    hits[["recommendationid", "appid", "review", "label", "note"]].to_csv(out_file, index=False)
    print(f"Exported {len(hits)} hits to {out_file}")
else:
    print(f"{out_file} already exists; not overwriting hand labels")

data/pilot_hits_labeled.csv already exists; not overwriting hand labels


## Hand-Check Results

In [7]:
# Hand check results
# Codes: G = generative AI, E = enemy/NPC/bot behavior, N = keyword matched but no AI meaning, ? = unclear
labeled = pd.read_csv("data/pilot_hits_labeled.csv")

assert len(labeled) == len(hits), "Every keyword hit should be in the labeled file"
assert labeled["label"].notna().all(), "Some hits are still unlabeled"
assert set(labeled["label"]) <= {"G", "E", "N", "?"}, f"Unexpected labels: {set(labeled['label'])}"

print(labeled["label"].value_counts())

def wilson_ci(k, n, z=1.96):
    """95% Wilson confidence interval for a proportion (works well for small counts)."""
    p = k / n
    center = (p + z**2 / (2*n)) / (1 + z**2 / n)
    half = z * np.sqrt(p*(1-p)/n + z**2 / (4*n**2)) / (1 + z**2 / n)
    return center - half, center + half

n_g = (labeled["label"] == "G").sum()
precision = n_g / len(labeled)
rate = n_g / len(pilot)
low, high = wilson_ci(n_g, len(pilot))
print(f"\nKeyword precision: {n_g} of {len(labeled)} hits ({precision:.0%}) were generative AI")
print(f"Estimated generative-AI rate: {rate:.1%} (95% CI {low:.1%} to {high:.1%})")

label
G    8
E    6
N    1
Name: count, dtype: int64

Keyword precision: 8 of 15 hits (53%) were generative AI
Estimated generative-AI rate: 2.6% (95% CI 1.3% to 5.0%)


## Investigate Edge Cases

In [8]:
# The non-G hits and their notes become examples for the labeling guide
for _, row in labeled[labeled["label"] != "G"].iterrows():
    print(f"[{row['label']}] {row['note'] if pd.notna(row['note']) else ''}\n   {row['review'][:150]}...\n")

[E] 
   This is not a fun game to play in its current state, there is some minor fun to be had with building a house and gear. But the human enemies are all S...

[E] 
   Just a generic survival wanna be game with stupid AI that just runs straight at you
the aiming is over sensitive no matter what settings you use ...fo...

[E] 
   (Update: It's better, but still way too buggy to recommend.)
Guns feels great, but the AI is so broken you'll just get aim botted from bushes before e...

[E] pretty sure it's E but AI mapping? What is that
   Well... let's start with the good, if you have a pretty beefy PC the game is gorgeous(def disable motion blur otherwise its a mess), The voice acting ...

[E] 
   I really wanted to like Dreadzone. It has a gritty, post-apocalyptic vibe concept that could’ve been a hit—but right out of the gate, the pacing felt ...

[E] 
   Let me start off by stating I actually really like the idea of this game a lot. The idea environment is pretty, but the sun rays a

## Pilot Decisions
- Complaints about enemy, NPC, or bot behavior ("the AI just runs at you") count as quality complaints, not generative AI complaints.
- Reviews are labeled by the reviewer's stated reason, so "AI random gen skins" counts as generative AI whether or not AI was actually used.
- "Slop" counts as generative AI only when AI is named or clearly implied.
- Keyword precision was 53%, so a keyword search alone is not reliable; this motivates the Claude classifier.
- These 312 reviews are the development set: the Claude prompt is tuned only on them, and they are removed from all test data.
- Bakery Cafe Simulator was in the pilot but later dropped from the final game list (only 23 negative reviews in the window).

---

# Part 2: Clean
**Purpose:** Clean the raw Steam review scrape, remove the pilot (development) reviews, and draw a random sample of up to 100 negative reviews per game for classification.

**Input:** `data/reviews_raw.csv` (raw scrape, Nov 4, 2026), `data/reviews_raw_replacements.csv` (2 replacement games), `data/games_final.csv` (30 games), `data/pilot.csv` (development set)

**Output:** `data/reviews_clean.csv`

**Summary:** 6,612 raw reviews plus 100 from 2 replacement games --> 2,638 cleaned reviews (1,197 disclosed, 1,441 control) from 30 games. Removed 89 reviews from the 2 replaced games, 200 pilot (dev set) reviews, and 19 non-English reviews, then randomly sampled up to 100 reviews per game (seed 42).

## Load the Data

In [9]:
#Read in the data and confirm
df = pd.read_csv("data/reviews_raw.csv")
games = pd.read_csv("data/games_final.csv")
pilot = pd.read_csv("data/pilot.csv")

print(f"Loaded reviews: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Loaded games: {games.shape[0]} rows")
print(f"Loaded pilot (dev set): {pilot.shape[0]} rows")

Loaded reviews: 6612 rows, 29 columns
Loaded games: 30 rows
Loaded pilot (dev set): 312 rows


## Explore the Data

In [10]:
#Explore the data structure
print("\nINFO:")
df.info()

print("\nDESCRIBE:")
print(df.describe().to_string())

print("\nDUPLICATES:")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate review IDs: {df['recommendationid'].duplicated().sum()}")

print("\nMISSING VALUES:")
print(df.isnull().sum())


INFO:
<class 'pandas.DataFrame'>
RangeIndex: 6612 entries, 0 to 6611
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   appid                          6612 non-null   int64  
 1   recommendationid               6612 non-null   int64  
 2   review                         6612 non-null   str    
 3   voted_up                       6612 non-null   bool   
 4   votes_up                       6612 non-null   int64  
 5   votes_funny                    6612 non-null   int64  
 6   weighted_vote_score            6612 non-null   float64
 7   timestamp_created              6612 non-null   int64  
 8   language                       6612 non-null   str    
 9   steam_purchase                 6612 non-null   bool   
 10  received_for_free              6612 non-null   bool   
 11  written_during_early_access    6612 non-null   bool   
 12  refunded                       6612 non-null   bool 

In [11]:
#Explore the data: content checks specific to this scrape

# Date range: should fall within Jan 1, 2024 – Dec 31, 2025
print("DATE RANGE:")
created = pd.to_datetime(df["timestamp_created"], unit="s")
print(f"Earliest: {created.min()}, Latest: {created.max()}")

# Scrape filters: should be all negative (voted_up False) and all English
print("\nSCRAPE FILTER CHECKS:")
print(df["voted_up"].value_counts(dropna=False))
print(df["language"].value_counts(dropna=False))

# Reviews per game (smallest first) and whether all 30 games returned reviews
print("\nREVIEWS PER GAME:")
per_game = df.groupby(["group", "name"]).size().sort_values()
print(per_game.to_string())
print(f"\nGames with reviews: {df['appid'].nunique()} of {len(games)}")
missing_games = set(games["appid"]) - set(df["appid"])
print(f"Games with no reviews: {missing_games if missing_games else 'none'}")

# Reviews by group (disclosed vs. control)
print("\nREVIEWS BY GROUP:")
print(df["group"].value_counts())

# Pilot reviews in the scrape (dev set; removed later so they aren't used for testing)
print("\nPILOT (DEV SET) OVERLAP:")
in_pilot = df["recommendationid"].isin(pilot["recommendationid"])
print(f"Reviews also in pilot.csv: {in_pilot.sum()}")
print(df[in_pilot].groupby("name").size().to_string())

# Review types that may need handling: empty, very short, checkbox templates, non-English text
print("\nREVIEW TEXT:")
text = df["review"].fillna("").str.strip()
words = text.str.split().str.len()
print(f"Empty reviews: {(text == '').sum()}")
print(f"Very short reviews (1-2 words): {(words.between(1, 2)).sum()}")
print(f"Checkbox-template reviews (contain ☐ or ☑): {text.str.contains('☐|☑').sum()}")
print(f"Mostly non-Latin text (under 50% Latin letters): "
      f"{(text.str.count(r'[A-Za-z]') / text.str.len().clip(lower=1) < 0.5).sum()}")

# Review length in words
print("\nWords per review:")
print(words.describe().round(1).to_string())

DATE RANGE:
Earliest: 2024-01-01 06:47:12, Latest: 2025-12-31 22:54:31

SCRAPE FILTER CHECKS:
voted_up
False    6612
Name: count, dtype: int64
language
english    6612
Name: count, dtype: int64

REVIEWS PER GAME:
group      name                               
disclosed  Night Club Simulator                    41
control    Pharmacy Simulator                      41
disclosed  Cinema Simulator 2025                   45
           Hotel Business Simulator                48
           Shawarma Legend                         56
           Esports Godfather                       56
           FACEMINER                               85
           Serum: Toxic Survival                   85
           Chernobylite 2: Exclusion Zone          87
control    Trash Goblin                           112
disclosed  Lost Rift                              119
control    Tavern Manager Simulator               124
           Ultimate Fishing Simulator 2           128
           Motel Manager Simulator    

## Exploration Observations
- 6,612 reviews from the 30 originally selected games, 29 columns. The two replacement games (Step 1) were scraped separately, so they appear under "Games with no reviews" here.
- The scrape filters worked! Every review is negative, tagged English, and dated Jan 1, 2024 – Dec 31, 2025.
- Disclosed games have about half as many reviews (2,250 vs. 4,362), consistent with the review gap found in prior research. Four control games hit the 500-review cap. Sampling up to 100 per game keeps large games from dominating.
- Counts can differ by one from the earlier summary run (e.g. Motel Manager Simulator: 131 vs. 132) because Steam data changes over time. All analysis uses this dated snapshot.
- 289 reviews overlap with the pilot (dev set): all of DREADZONE (100 of 187), TRIBES 3 (100 of 464), Hotel Business Simulator (48 of 48), and Night Club Simulator (41 of 41). Removing them leaves Hotel Business and Night Club with no test reviews.
    - Decision: both are replaced with two other low-rated AI-disclosed games (Step 1), so the 312 pilot reviews stay a clean dev set.
- 277 reviews (4.2%) are only 1–2 words, and 8 are checkbox templates. These are kept: they are real negative reviews, labeled other if they give no reason.
- 53 reviews (0.8%) are mostly non-Latin characters. These could be other languages tagged as English, or ASCII art; they are inspected before deciding whether to remove them.
- Reviews have a median of 48 words (mean 100, max 1,541). The long tail matters for the cost of the Claude step.

---

## Cleaning Steps

In [12]:
df_clean = df.copy()
print("Starting rows:", len(df_clean))

Starting rows: 6612


## Step 1: Replace Games With No Test Reviews

In [13]:
# 1. Replace games whose reviews are all in the pilot (dev set)
n_before = len(df_clean)
print("Step 1 (replace games) - Starting rows:", n_before)

# Replacement scrape (already run on Nov 4, 2026):
# %run steam_reviews_scraper.py --appid-file data/game_selection/games_replacements.csv --max-per-game 500 --out data/reviews_raw_replacements.csv

dropped_ids = [2738870, 2943670]  # Hotel Business Simulator, Night Club Simulator
replacements = pd.read_csv("data/reviews_raw_replacements.csv")
expected_drop = df_clean["appid"].isin(dropped_ids).sum()

df_clean = df_clean[~df_clean["appid"].isin(dropped_ids)]
df_clean = pd.concat([df_clean, replacements], ignore_index=True)
n_after = len(df_clean)

# Assert the swap: old games gone, new reviews added, review set matches the 30 final games
print("Step 1 (replace games) - Ending rows:", n_after,
      f"(dropped {expected_drop}, added {len(replacements)})")
assert n_after == n_before - expected_drop + len(replacements)
assert not df_clean["appid"].isin(dropped_ids).any(), "Dropped games should be gone"
assert set(df_clean["appid"]) == set(games["appid"]), "Reviews should cover exactly the 30 final games"
assert set(replacements.columns) == set(df.columns), "Replacement scrape should have the same columns"
assert df_clean["recommendationid"].is_unique, "Review IDs should be unique"

# Why: All in-window reviews for Hotel Business Simulator and Night Club Simulator were in the pilot,
# which is the dev set and can't be used for testing. Replacing them with two low-rated AI-disclosed
# management sims (REAL ESTATE Simulator, The Last FARMER) keeps 5 games in each rating tier and the same genre mix.

Step 1 (replace games) - Starting rows: 6612
Step 1 (replace games) - Ending rows: 6623 (dropped 89, added 100)


## Step 2: Parse Review Dates

In [14]:
# 2. Parse review dates
n_before = len(df_clean)
print("Step 2 (dates) - Starting rows:", n_before)

df_clean["created"] = pd.to_datetime(df_clean["timestamp_created"], unit="s")

n_after = len(df_clean)
print("Step 2 (dates) - Ending rows:", n_after)
print(f"Date range: {df_clean['created'].min()} to {df_clean['created'].max()}")

# Assert row count unchanged, every date parsed, and all dates inside the study window
assert n_after == n_before, "Parsing dates shouldn't change row count"
assert df_clean["created"].notna().all(), "Some dates failed to parse"
assert df_clean["created"].between("2024-01-01", "2025-12-31 23:59:59").all(), "Reviews outside the study window"

# Why: created was read from the CSV as text, so it can't be sorted or grouped by month. Parsing it from the
# Unix timestamp gives a true datetime and confirms Steam's date filter also held for the replacement scrape.

Step 2 (dates) - Starting rows: 6623
Step 2 (dates) - Ending rows: 6623
Date range: 2024-01-01 06:47:12 to 2025-12-31 22:54:31


## Step 3: Validate Scrape Filters

In [15]:
# 3. Validate scrape filters
n_before = len(df_clean)
print("Step 3 (filters) - Starting rows:", n_before)
print(df_clean["voted_up"].value_counts(dropna=False))
print(df_clean["language"].value_counts(dropna=False))
n_after = len(df_clean)
print("Step 3 (filters) - Ending rows:", n_after)

# Assert every review is negative and English, including the replacement scrape
assert n_after == n_before
assert (df_clean["voted_up"] == False).all(), "Found a positive review"
assert (df_clean["language"] == "english").all(), "Found a non-English review"

# Why: The study only covers English negative reviews. If Steam's API ever returned other reviews, they would
# silently mix into the sample, so this confirms the filters held for both scrapes.

Step 3 (filters) - Starting rows: 6623
voted_up
False    6623
Name: count, dtype: int64
language
english    6623
Name: count, dtype: int64
Step 3 (filters) - Ending rows: 6623


## Step 4: Remove Pilot (Dev Set) Reviews

In [16]:
# 4. Remove pilot (dev set) reviews
n_before = len(df_clean)
print("Step 4 (pilot) - Starting rows:", n_before)

in_pilot = df_clean["recommendationid"].isin(pilot["recommendationid"])
expected_drop = in_pilot.sum()
print(df_clean[in_pilot].groupby("name").size().to_string())
df_clean = df_clean[~in_pilot].reset_index(drop=True)

n_after = len(df_clean)
print("Step 4 (pilot) - Ending rows:", n_after, f"(dropped {n_before - n_after})")

# Assert the expected rows were removed and no pilot review remains
assert n_after == n_before - expected_drop
assert not df_clean["recommendationid"].isin(pilot["recommendationid"]).any(), "Pilot reviews remain"

# Why: The pilot reviews were used to design the keyword search and will be used to tune the Claude prompt.
# Testing on them would overstate how well both methods work, so they are kept out of all test data.

Step 4 (pilot) - Starting rows: 6623
name
DREADZONE           100
TRIBES 3: Rivals    100
Step 4 (pilot) - Ending rows: 6423 (dropped 200)


## Inspect Possibly Non-English Reviews

In [17]:
# Look at reviews with under 50% Latin letters before deciding whether to remove them
text = df_clean["review"].str.strip()
latin_share = text.str.count(r"[A-Za-z]") / text.str.len().clip(lower=1)
non_latin = df_clean[latin_share < 0.5]

print(f"Mostly non-Latin reviews: {len(non_latin)}")
print(non_latin.groupby(["group", "name"]).size().to_string())
print()
for review in non_latin["review"].head(25):
    print("-", review[:120].replace("\n", " "))

Mostly non-Latin reviews: 51
group      name                               
control    Aloft                                   1
           Drug Dealer Simulator 2                10
           FBC: Firebreak                          4
           Farming Simulator 25                    3
           Kebab Chefs! - Restaurant Simulator     2
           Killing Floor 3                         5
           Sengoku Dynasty                         4
           Sniper Elite: Resistance                3
           Tactical Breach Wizards                 2
           Tavern Manager Simulator                2
           Trash Goblin                            1
           Ultimate Fishing Simulator 2            3
disclosed  DREADZONE                               2
           Esports Godfather                       2
           FACEMINER                               1
           Serum: Toxic Survival                   1
           Shawarma Legend                         3
           TRIBES 3: Ri

## Inspection Observations
- 51 reviews have under 50% Latin letters (2 fewer than in the raw scrape, since those were removed with the pilot or replaced games). They are three different kinds:
    - Written in another language but tagged English: Russian, Ukrainian, Korean, Thai, Arabic (e.g. "не работают сервера! возврат").
    - No words at all: punctuation, emoji, or ASCII art (".", ":(", "💩", "-1").
    - English with many symbols, caught by mistake ("?????? what is the problem of this game").
- The 50% rule mixes these up, so it isn't used to drop reviews. Step 5 drops only reviews where most letters are from another alphabet, which targets non-English reviews.
- Reviews with no letters are kept, consistent with keeping very short reviews: they are real negative reviews and are labeled other.
- Most flagged reviews come from control games (e.g. Drug Dealer Simulator 2: 10), so Step 5 removes slightly more control reviews; the effect is small.

## Step 5: Remove Non-English Reviews

In [18]:
# 5. Remove non-English reviews (written mainly in another alphabet)
n_before = len(df_clean)
print("Step 5 (language) - Starting rows:", n_before)

text = df_clean["review"].fillna("")
letters = text.apply(lambda s: sum(ch.isalpha() for ch in s))   # letters in any alphabet; symbols, emoji, ASCII art don't count
latin_letters = text.str.count(r"[A-Za-zÀ-ɏ]")                  # Latin letters, including accented ones
other_script = (letters > 0) & ((letters - latin_letters) / letters.clip(lower=1) > 0.5)

expected_drop = other_script.sum()
print(f"Non-English reviews: {expected_drop}")
print(df_clean[other_script].groupby("group").size().to_string())
for review in df_clean.loc[other_script, "review"].head(10):
    print("-", review[:80].replace("\n", " "))

df_clean = df_clean[~other_script].reset_index(drop=True)
n_after = len(df_clean)
print("Step 5 (language) - Ending rows:", n_after, f"(dropped {n_before - n_after})")
print(f"Kept reviews with no letters (punctuation, emoji, ASCII art): {(letters[~other_script] == 0).sum()}")

# Assert the expected rows were removed, and only from the 51 flagged in the inspection
assert n_after == n_before - expected_drop
assert expected_drop <= 51, "Should drop no more than the reviews flagged in the inspection"

# Why: The labeling guide, hand labels, and keyword baseline are English only, so reviews written in another language
# can't be labeled consistently. Steam tags them English because that's the reviewer's setting, not the text's language.
# Counting letters only (not symbols) keeps English reviews with lots of punctuation or emoji.
# Non-English reviews written in the Latin alphabet (e.g. Spanish) aren't caught by this rule.

Step 5 (language) - Starting rows: 6423
Non-English reviews: 19
group
control      15
disclosed     4
- у всего один эффект, не интересно, шедуле лучше
 [*]лечето классов персонажи как в героических шутанах
- English. Not many new features except added dodge. Many more flaws. Alpha versio
- Мало контента и долгая прокачка.  Хочется пробовать новые скилы в прокачке, но к
- 병맛좀 곁들인 미완성적 b급 낚시게임
- не работают сервера! возврат
- Один вопрос: "Почему это существует, и кто вообще посчитал разумным пропустить э
- ตอนแรกก็บัคเยอะอยู่แล้ว แทนที่อัพเดตใหม่จะดีขึ้น กากเหมือนเดิม ไม่เข้าใจ เกมก็เส
- بتضل تعلق
 Контента как такого-го мало, только если за
Step 5 (language) - Ending rows: 6404 (dropped 19)
Kept reviews with no letters (punctuation, emoji, ASCII art): 19


**Step 5 Results:** 19 reviews dropped (15 control, 4 disclosed) and 19 no-letter reviews kept. One bilingual review (English summary followed by Ukrainian) was dropped because most of its letters are Ukrainian; its English part was a short summary, so little information is lost.

## Step 6: Add Computed Columns

In [19]:
# 6. Add computed columns
n_before = len(df_clean)
print("Step 6 (computed columns) - Starting rows:", n_before)

df_clean["word_count"] = df_clean["review"].str.split().str.len()
df_clean["playtime_at_review_hrs"] = (df_clean["author_playtime_at_review_min"] / 60).round(1)
df_clean["playtime_forever_hrs"] = (df_clean["author_playtime_forever_min"] / 60).round(1)

n_after = len(df_clean)
print("Step 6 (computed columns) - Ending rows:", n_after)
print(df_clean[["word_count", "playtime_at_review_hrs"]].describe().round(1).to_string())

# Assert row count unchanged and the new columns are complete and valid
assert n_after == n_before, "Adding columns should not change row count"
assert df_clean["word_count"].notna().all() and (df_clean["word_count"] >= 1).all()
assert (df_clean["playtime_at_review_hrs"] >= 0).all()

# Why: Word count is needed to estimate Claude's cost and to check whether very short reviews are labeled differently.
# Playtime in hours is easier to read than minutes and shows whether reviewers played much before reviewing.

Step 6 (computed columns) - Starting rows: 6404
Step 6 (computed columns) - Ending rows: 6404
       word_count  playtime_at_review_hrs
count      6404.0                  6404.0
mean        101.0                    29.1
std         159.7                   122.3
min           1.0                     0.1
25%          18.0                     1.5
50%          49.0                     5.3
75%         117.0                    19.4
max        1541.0                  4254.7


## Step 7: Sample Up to 100 Reviews per Game

In [20]:
# 7. Randomly sample up to 100 reviews per game
n_before = len(df_clean)
print("Step 7 (sample) - Starting rows:", n_before)

per_game_before = df_clean.groupby("appid").size()
expected = per_game_before.clip(upper=100)

# Shuffle with a fixed seed, then keep the first 100 of each game
df_clean = (df_clean.sample(frac=1, random_state=42)
            .groupby("appid").head(100)
            .sort_values(["group", "name", "created"])
            .reset_index(drop=True))

n_after = len(df_clean)
per_game_after = df_clean.groupby("appid").size()
print("Step 7 (sample) - Ending rows:", n_after, f"(dropped {n_before - n_after})")
print(df_clean.groupby("group").size().to_string())

# Assert every game has at most 100, games under 100 kept all their reviews, and all 30 games remain
assert (per_game_after <= 100).all(), "A game has more than 100 reviews"
assert per_game_after.equals(expected.loc[per_game_after.index]), "Games under 100 should keep every review"
assert n_after == expected.sum()
assert df_clean["appid"].nunique() == 30, "All 30 games should remain"

# Why: Some games have hundreds of negative reviews and others under 50. Without capping, a few large control games
# would dominate the results. The random sample (seed 42) is reproducible and avoids the newest-first order Steam returns.

Step 7 (sample) - Starting rows: 6404
Step 7 (sample) - Ending rows: 2638 (dropped 3766)
group
control      1441
disclosed    1197


## Step 8: Save the Cleaned Data

In [21]:
# 8. Save the cleaned data and confirm it reloads correctly
df_clean.to_csv("data/reviews_clean.csv", index=False)
check = pd.read_csv("data/reviews_clean.csv")
print(f"Saved data/reviews_clean.csv: {check.shape[0]} rows, {check.shape[1]} columns")

assert len(check) == len(df_clean), "Saved file should have the same rows"
assert check["recommendationid"].is_unique, "Review IDs should be unique in the saved file"

Saved data/reviews_clean.csv: 2638 rows, 32 columns


## Balance Table

In [22]:
# Compare the disclosed and control groups at the game level
games["is_sim"] = games["genre"].str.contains("sim")
balance = games.groupby("group").agg(
    games=("appid", "count"),
    sim_games=("is_sim", "sum"),
    median_price=("base_price_usd", "median"),
    median_pct_positive=("pct_positive_alltime", "median"),
    median_english_reviews=("total_reviews_alltime", "median"),
    earliest_release=("release", "min"),
    latest_release=("release", "max"),
)
balance["sampled_reviews"] = df_clean.groupby("group").size()
print(balance.T.to_string())
print()
print(pd.crosstab(games["group"], games["rating_tier"]))

group                   control disclosed
games                        15        15
sim_games                     8         9
median_price              19.99     19.99
median_pct_positive        79.7      73.0
median_english_reviews   2324.0     761.0
earliest_release        2024-01   2024-03
latest_release          2025-07   2025-09
sampled_reviews            1441      1197

rating_tier  high  low  mid
group                      
control         6    4    5
disclosed       5    5    5


## Cleaning Observations
- **Final dataset:** 2,638 reviews from 30 games (1,197 disclosed, 1,441 control). The disclosed group is smaller because 9 of its games had fewer than 100 reviews, while 14 of 15 control games reached the cap.
- **Many reviewers played briefly:** a quarter played under 1.5 hours before reviewing (median 5.3 hours), inside Steam's 2-hour refund window. Playtime is very skewed (0.1 to 4,255 hours), so medians are used rather than means.
- **The groups are well matched:** on number of games, genre mix (9 vs. 8 sims), median price ($19.99 for both), release window, and rating tiers (5/5/5 vs. 6/5/4).
- **Two differences are expected and not matched on purpose:** disclosed games are rated lower (73.0% vs. 79.7% positive) and have about a third as many English reviews (761 vs. 2,324). Both may be part of the AI penalty, so matching on them would hide the effect being studied. Because lower-rated games may draw more quality complaints regardless of AI, results are also compared within rating tiers.

## Minimum Detectable Difference

In [23]:
# Smallest difference in per-game complaint rates (disclosed vs. control) detectable with 15 vs. 15 games
# Two-sided test, alpha = 0.05, 80% power. The true game-to-game spread is unknown until labeling, so several are shown.
n_per_group = 15
df_t = 2 * n_per_group - 2
multiplier = stats.t.ppf(0.975, df_t) + stats.t.ppf(0.80, df_t)

mdd = pd.DataFrame({"sd_between_games": [0.05, 0.10, 0.15, 0.20]})
mdd["min_detectable_diff"] = (multiplier * mdd["sd_between_games"] * np.sqrt(2 / n_per_group)).round(3)
print(f"Multiplier: {multiplier:.2f}  (MDD ≈ {multiplier * np.sqrt(2 / n_per_group):.2f} × SD between games)")
print(mdd.to_string(index=False))

Multiplier: 2.90  (MDD ≈ 1.06 × SD between games)
 sd_between_games  min_detectable_diff
             0.05                0.053
             0.10                0.106
             0.15                0.159
             0.20                0.212


## Minimum Detectable Difference Observations
- With 15 games per group, the smallest difference in per-game complaint rates we can reliably detect is about 1.06 × the spread between games (two-sided test, α = 0.05, 80% power).
- If quality complaint rates vary by about 10 points from game to game, only a gap of about 11 points between disclosed and control games would be detectable. Smaller real differences could be missed.
- Generative-AI complaints are rare (about 2.6% in the pilot), so their game-to-game spread is likely small too, which makes a gap of a few points detectable. Control games should be near zero.
- The real spread is unknown until labeling, so this is recomputed with observed values in Part 5. A non-significant result is reported as "no detectable difference," not "no difference."

---

# Part 3: Keyword Baseline

**Purpose:** Flag reviews that match the keyword pattern locked in the pilot. This is the simple, non-AI method the Claude classifier is compared against.

**Input:** `data/reviews_clean.csv` 

**Output:** `data/keyword_flags.csv`

## Load the Cleaned Data

In [24]:
# Load the cleaned reviews
reviews = pd.read_csv("data/reviews_clean.csv")
print(f"Loaded: {reviews.shape[0]} rows, {reviews.shape[1]} columns")

assert len(reviews) == 2638, f"Expected 2,638 cleaned reviews, found {len(reviews)}"
assert reviews["recommendationid"].is_unique

Loaded: 2638 rows, 32 columns


## Apply the Keyword Pattern

In [25]:
# Flag reviews matching the locked pilot pattern (AI_PATTERN from Part 1, unchanged)
reviews["keyword_flag"] = reviews["review"].str.contains(AI_PATTERN, case=False, regex=True)

flags = reviews.groupby("group")["keyword_flag"].agg(flagged="sum", reviews="count")
flags["rate"] = (flags["flagged"] / flags["reviews"]).round(3)
print(flags.to_string())
print(f"\nTotal flagged: {reviews['keyword_flag'].sum()} of {len(reviews)}")

# Save the flags for building the labeling sheet and for evaluation
reviews[["recommendationid", "keyword_flag"]].to_csv("data/keyword_flags.csv", index=False)
assert pd.read_csv("data/keyword_flags.csv").shape[0] == len(reviews)

           flagged  reviews   rate
group                             
control         67     1441  0.046
disclosed      169     1197  0.141

Total flagged: 236 of 2638


## Keyword Baseline Observations
- **236 of 2,638 reviews (8.9%) match the keyword pattern:** 169 of 1,197 disclosed (14.1%) and 67 of 1,441 control (4.6%).
- The disclosed rate is much higher than the pilot's 4.8%, which came from only 5 games. These are keyword matches, not confirmed generative-AI complaints: in the pilot only 53% of matches were.
- Control games also match, likely mostly mentions of enemy or NPC behavior in shooters and survival games.
- Flagged reviews are not printed, so they can be labeled blind in round 1.

## Round 1 Labeling Sheet

In [26]:
# Build the round 1 labeling sheet: every keyword-flagged review plus 100 random unflagged reviews
n_flagged = reviews["keyword_flag"].sum()
flagged = reviews[reviews["keyword_flag"]]
unflagged_sample = reviews[~reviews["keyword_flag"]].sample(n=100, random_state=42)

# Shuffle so flagged and unflagged reviews are mixed, then mark a random 100 for the second labeler
sheet = pd.concat([flagged, unflagged_sample]).sample(frac=1, random_state=7).reset_index(drop=True)
sheet["sheet_id"] = range(1, len(sheet) + 1)
sheet["double_label"] = False
sheet.loc[sheet.sample(n=100, random_state=11).index, "double_label"] = True

print(f"Round 1 sheet: {len(sheet)} reviews ({n_flagged} keyword-flagged + 100 random unflagged)")
print(f"Marked for second labeler: {sheet['double_label'].sum()}")

assert len(sheet) == n_flagged + 100
assert sheet["recommendationid"].is_unique
assert sheet["double_label"].sum() == 100
assert not sheet["recommendationid"].isin(pilot["recommendationid"]).any(), "No pilot (dev set) reviews"

# Blind sheets show only an ID and the review text; the key file holds group and keyword flag for later
# Files are only written once, so re-running the notebook never erases labels
blank = {"reason": "", "ai_target": "", "unsure": "", "note": ""}
blind = sheet[["sheet_id", "review"]].assign(**blank)
files = {
    "data/labeling_key.csv": sheet[["sheet_id", "recommendationid", "appid", "group", "keyword_flag", "double_label"]],
    "data/labels_round1_neha.csv": blind,
    "data/labels_round1_second.csv": blind[sheet["double_label"].values],
}
for path, frame in files.items():
    if not os.path.exists(path):
        frame.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"Wrote {path} ({len(frame)} rows)")
    else:
        print(f"{path} already exists; not overwriting")

Round 1 sheet: 336 reviews (236 keyword-flagged + 100 random unflagged)
Marked for second labeler: 100
data/labeling_key.csv already exists; not overwriting
data/labels_round1_neha.csv already exists; not overwriting
data/labels_round1_second.csv already exists; not overwriting


----

# Part 4: Hand Labels (Test Set)
**Purpose:** Combine the round 1 hand labels, check them, and measure agreement between the two labelers.

**Input:** `data/labels_round1_neha_part1.csv`, `data/labels_round1_neha_part2.xlsx`, `data/labels_round1_second.csv`, `data/labeling_key.csv`

**Output:** `data/labels_round1_neha_combined.csv`, `data/disagreements_round1.csv`

## Combine Labels From Two Files

In [36]:
# Combine labels from two working files (labeling was split across two laptops)
LABELS = {"GENAI", "QUALITY", "BOTH", "OTHER", "SKIP"}

def load_labels(path):
    # Reads .csv or .xlsx; drops blank rows that Excel adds around reviews with line breaks
    if path.endswith(".xlsx"):
        f = pd.read_excel(path, dtype=str).fillna("")
    else:
        f = pd.read_csv(path, dtype=str, keep_default_na=False)
    for col in ["sheet_id", "reason", "unsure", "note"]:
        f[col] = f[col].fillna("").astype(str).str.strip()

    blank_id = f["sheet_id"] == ""
    if blank_id.any():
        stray = blank_id & (f["reason"] != "")
        print(f"{path}: dropped {blank_id.sum()} rows with no sheet_id ({stray.sum()} had a label)")
        assert not stray.any(), f"Labels on rows with no sheet_id:\n{f[stray].to_string()}"
        f = f[~blank_id]

    f["sheet_id"] = pd.to_numeric(f["sheet_id"]).astype(int)
    f["reason"] = f["reason"].str.upper()
    assert f["sheet_id"].is_unique, f"Duplicate sheet_ids in {path}"
    return f.set_index("sheet_id")

key = pd.read_csv("data/labeling_key.csv")
p1 = load_labels("data/labels_round1_neha_part1.csv")
p2 = load_labels("data/labels_round1_neha_part2.xlsx")
assert set(p1.index) == set(key["sheet_id"]), "Part 1 rows don't match the labeling sheet"
assert set(p2.index) == set(key["sheet_id"]), "Part 2 rows don't match the labeling sheet"
p2 = p2.reindex(p1.index)

# Check for reviews labeled differently in the two files
both_labeled = (p1["reason"] != "") & (p2["reason"] != "")
conflicts = both_labeled & (p1["reason"] != p2["reason"])
print(f"Labeled in part 1: {(p1['reason'] != '').sum()}, part 2: {(p2['reason'] != '').sum()}, both: {both_labeled.sum()}")
print(f"Conflicting labels: {conflicts.sum()}")
if conflicts.any():
    print(p1[conflicts][["reason"]].join(p2[conflicts][["reason"]], rsuffix="_part2").to_string())

# Take each label from whichever file has it
neha = p1.copy()
for col in ["reason", "unsure", "note"]:
    neha[col] = p1[col].where(p1["reason"] != "", p2[col])
neha = neha.reset_index()

# Fixes found by the assert checks below (recorded here rather than edited in the files)
fixes = {
    147: ("GENAI", "missed while labeling; ethical objection to AI voice acting"),
    21: ("QUALITY", "originally 'BOTH OR QUALITY'; reviewer dismisses others' 'AI slop' claim and only makes quality complaints"),
}
for sid, (label, why) in fixes.items():
    neha.loc[neha["sheet_id"] == sid, ["reason", "note"]] = [label, why]
    print(f"Fixed sheet_id {sid}: {label} ({why})")

print(f"\nCombined: {len(neha)} rows")
print(neha["reason"].value_counts().to_string())

# Assert every review has exactly one valid label and nothing conflicts
assert not conflicts.any(), "Resolve conflicting labels before continuing"
assert (neha["reason"] != "").all(), f"Unlabeled sheet_ids: {neha.loc[neha['reason'] == '', 'sheet_id'].tolist()}"
assert set(neha["reason"]) <= LABELS, f"Unexpected labels: {set(neha['reason']) - LABELS}"

# Check the review text wasn't shifted or altered while editing in Excel/Sheets
clean_text = lambda s: s.fillna("").astype(str).str.split().str.join(" ")
original = key.merge(reviews[["recommendationid", "review"]], on="recommendationid").set_index("sheet_id")["review"]
matches = (clean_text(neha.set_index("sheet_id")["review"]) == clean_text(original.reindex(neha["sheet_id"]))).mean()
print(f"Review text matches the original sheet: {matches:.1%}")
assert matches > 0.99, "Review text changed; rows may have shifted while editing"

neha.to_csv("data/labels_round1_neha_combined.csv", index=False)
print("Saved data/labels_round1_neha_combined.csv")

data/labels_round1_neha_part2.xlsx: dropped 336 rows with no sheet_id (0 had a label)
Labeled in part 1: 234, part 2: 120, both: 19
Conflicting labels: 0
Fixed sheet_id 147: GENAI (missed while labeling; ethical objection to AI voice acting)
Fixed sheet_id 21: QUALITY (originally 'BOTH OR QUALITY'; reviewer dismisses others' 'AI slop' claim and only makes quality complaints)

Combined: 336 rows
reason
QUALITY    182
GENAI       72
BOTH        69
OTHER       12
SKIP         1
Review text matches the original sheet: 100.0%
Saved data/labels_round1_neha_combined.csv


## Agreement Between Labelers

In [41]:
# Agreement on the 100 double-labeled reviews
second = load_labels("data/labels_round1_tomas.csv").reset_index()
second = second[second["reason"] != ""]
assert len(second) == 100, f"Second labeler's file should have 100 labeled rows, found {len(second)}"
assert set(second["reason"]) <= LABELS, f"Unexpected labels: {set(second['reason']) - LABELS}"

pair = second[["sheet_id", "reason"]].merge(neha[["sheet_id", "reason"]], on="sheet_id", suffixes=("_tomas", "_neha"))
assert len(pair) == 100, "All 100 double-labeled reviews should match"

raw = (pair["reason_tomas"] == pair["reason_neha"]).mean()
kappa = cohen_kappa_score(pair["reason_neha"], pair["reason_tomas"])
print(f"Raw agreement: {raw:.1%}   Cohen's kappa: {kappa:.2f}\n")
print(pd.crosstab(pair["reason_neha"], pair["reason_tomas"], rownames=["Neha"], colnames=["Tomas"]))

# Per-label agreement (kappa can look low when one label is rare)
print("\nPer-label agreement:")
for label in sorted(LABELS):
    a, b = pair["reason_neha"] == label, pair["reason_tomas"] == label
    if a.sum() + b.sum() > 0:
        print(f"  {label:8s} {2 * (a & b).sum() / (a.sum() + b.sum()):.0%}  (Neha {a.sum()}, Tomas {b.sum()})")

disagree = (pair[pair["reason_tomas"] != pair["reason_neha"]]
            .merge(neha[["sheet_id", "review", "note"]], on="sheet_id"))
disagree.to_csv("data/disagreements_round1.csv", index=False)
print(f"\nDisagreements to settle: {len(disagree)} (saved to data/disagreements_round1.csv)")

Raw agreement: 77.0%   Cohen's kappa: 0.64

Tomas    BOTH  GENAI  OTHER  QUALITY  SKIP
Neha                                      
BOTH       19      2      0        4     0
GENAI       4     15      0        1     0
OTHER       0      0      1        1     0
QUALITY     9      1      1       41     0
SKIP        0      0      0        0     1

Per-label agreement:
  BOTH     67%  (Neha 25, Tomas 32)
  GENAI    79%  (Neha 20, Tomas 18)
  OTHER    50%  (Neha 2, Tomas 2)
  QUALITY  83%  (Neha 52, Tomas 47)
  SKIP     100%  (Neha 1, Tomas 1)

Disagreements to settle: 23 (saved to data/disagreements_round1.csv)


## Settle Disagreements

In [44]:
# Apply consensus labels for the double-labeled reviews where the labelers disagreed
resolved = pd.read_csv("data/disagreements_round1_resolved.csv", dtype=str, keep_default_na=False)
resolved["sheet_id"] = resolved["sheet_id"].astype(int)
resolved["consensus"] = resolved["consensus"].str.strip().str.upper()

assert len(resolved) == len(disagree), "Every disagreement should have a consensus label"
assert (resolved["consensus"] != "").all(), "Some disagreements have no consensus label"
assert set(resolved["consensus"]) <= LABELS, f"Unexpected labels: {set(resolved['consensus']) - LABELS}"

final = neha.copy()
final = final.merge(resolved[["sheet_id", "consensus", "why"]], on="sheet_id", how="left")
changed = final["consensus"].notna() & (final["consensus"] != final["reason"])
final.loc[final["consensus"].notna(), "reason"] = final["consensus"]
print(f"Consensus changed {changed.sum()} of Neha's labels ({len(resolved)} disagreements settled)")
print(final["reason"].value_counts().to_string())

tomas_changed = (resolved["consensus"] != resolved["reason_tomas"]).sum()
neither = ((resolved["consensus"] != resolved["reason_tomas"]) & (resolved["consensus"] != resolved["reason_neha"])).sum()
print(f"Consensus changed {tomas_changed} of Tomas's labels; {neither} went to a third label")

final = final.drop(columns=["consensus"])
final.to_csv("data/labels_round1_final.csv", index=False)
print("Saved data/labels_round1_final.csv")

Consensus changed 5 of Neha's labels (23 disagreements settled)
reason
QUALITY    184
BOTH        71
GENAI       69
OTHER       11
SKIP         1
Consensus changed 19 of Tomas's labels; 1 went to a third label
Saved data/labels_round1_final.csv


## Agreement: Observations
- **Two labelers agreed on 77 of 100 reviews (Cohen's kappa = 0.64)**, "substantial" agreement on the standard scale, above our 0.6 threshold.
- **On the study's key distinction, generative-AI complaint (GENAI or BOTH) vs. not, agreement was 85%** (kappa ≈ 0.70; positive agreement on the AI class 84%).
- **Most disagreements were QUALITY vs. BOTH (13 of 23):** reviews where the AI mention was mild or excused, or where it was unclear whether a complaint stood apart from the AI. The second labeler chose BOTH more often (32 vs. 25).
- **These disagreements led to guide v4:** any criticism of AI-made content counts unless explicitly excused, and BOTH requires a complaint that stands without the AI.
- **The 23 disagreements were settled together.** The final label kept the first labeler's choice in 18 and changed it in 5. Kappa is reported from the independent labels, before discussion.
- **Domain familiarity mattered:** many disagreements involved enemy or NPC "AI" (game behavior) labeled as generative AI. The second labeler was less familiar with gaming terminology, which suggests reviewers need training on this distinction; it is the same confusion that limits the keyword baseline.

---

# Part 5: Claude Classifier
**Purpose:** Label every review with Claude using the same labeling guide as the human labelers, after checking the prompt on the pilot (dev set) only. Follows the Class 11 LLM classifier setup (Sonnet 4.6, XML-tagged prompt, JSON output with confidence and evidence).

**Input:** `LABELING_GUIDE.md`, `data/pilot.csv`, `data/reviews_clean.csv` **Output:** `data/claude_prompt.txt`, `data/claude_labels_dev.csv`, `data/claude_labels.csv`

## Setup and Prompt

In [62]:
# Load API key from the .env file (never committed)
load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=api_key, max_retries=6)

MODEL = "claude-sonnet-4-6"
RUN_BY = "Neha"  # Why: the Documentation rule logs who ran each call
INPUT_RATE = 3.00 / 1_000_000        # $ per input token (Sonnet 4.6)
OUTPUT_RATE = 15.00 / 1_000_000      # $ per output token
CACHE_READ_RATE = 0.30 / 1_000_000   # $ per cached input token (the repeated labeling guide)
CACHE_WRITE_RATE = 3.75 / 1_000_000  # $ per token to write the cache

guide = open("LABELING_GUIDE.md", encoding="utf-8").read()
GUIDE_VERSION = guide.split("(v", 1)[1].split(")", 1)[0]

# XML tags separate the task rules from the labeling guide, as in class
SYSTEM_PROMPT = f"""<task>
Classify one negative Steam review by the reviewer's stated reason for not recommending the game.
Follow the labeling guide exactly; it is the same guide the human labelers used.
Return ONLY valid JSON, no other text:
{{"reason": "GENAI|QUALITY|BOTH|OTHER|SKIP", "confidence": "high|medium|low", "evidence": "key phrase", "english": true}}
- evidence: the short phrase from the review that decided the label
- english: true if the review is mainly in English or its English part states a reason, otherwise false
</task>

<labeling_guide>
{guide}
</labeling_guide>"""

with open("data/claude_prompt.txt", "w", encoding="utf-8") as f:
    f.write(SYSTEM_PROMPT)
print(f"Model: {MODEL} | Labeling guide v{GUIDE_VERSION} | Prompt saved to data/claude_prompt.txt")

Model: claude-sonnet-4-6 | Labeling guide v4 | Prompt saved to data/claude_prompt.txt


## Classifier Functions

In [63]:
def classify_text(text):
    """Classify a single review, tracking tokens and time."""
    start = time.time()
    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        system=[{"type": "text", "text": SYSTEM_PROMPT, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": f"Classify:\n\n{text}"}]
    )
    elapsed = time.time() - start
    raw = response.content[0].text.strip()
    # the model often wraps its JSON in ```json fences; keep only the {...} object
    start_brace, end_brace = raw.find("{"), raw.rfind("}")
    if start_brace != -1 and end_brace != -1:
        raw = raw[start_brace:end_brace + 1]

    try:
        parsed = json.loads(raw)
        parsed["reason"] = str(parsed["reason"]).strip().upper()
        if parsed["reason"] not in LABELS:
            raise ValueError
    except (json.JSONDecodeError, KeyError, ValueError):
        parsed = {"reason": "parse_error", "confidence": "low", "evidence": "", "english": None}

    return {
        **parsed,
        "model_version": response.model,             # the exact version the API served
        "raw_output": response.content[0].text,      # full reply before parsing
        "called_at": pd.Timestamp.now(tz="UTC").isoformat(),
        "run_by": RUN_BY,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "cache_read_tokens": getattr(response.usage, "cache_read_input_tokens", 0) or 0,
        "cache_write_tokens": getattr(response.usage, "cache_creation_input_tokens", 0) or 0,
        "response_time_s": round(elapsed, 2),
    }


def run_batch(frame, out_path, workers=5):
    """Classify every review in frame, a few at a time, saving every 50 so an interrupted run can resume."""
    done = pd.read_csv(out_path) if os.path.exists(out_path) else pd.DataFrame(columns=["recommendationid", "reason"])
    done_ids = set(done.loc[done["reason"].isin(LABELS), "recommendationid"])
    todo = frame[~frame["recommendationid"].isin(done_ids)]
    print(f"Classifying {len(todo)} reviews ({len(done_ids)} already done)...")

    def one(row):
        result = classify_text(row.review)
        result["recommendationid"] = row.recommendationid
        return result

    llm_results, errors = [], 0
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(one, row) for row in todo.itertuples()]
        for i, fut in enumerate(as_completed(futures), 1):
            try:
                llm_results.append(fut.result())
            except Exception as e:
                errors += 1
                if errors <= 3:
                    print(f"  Error: {e}")
            if llm_results and (len(llm_results) >= 50 or i == len(futures)):
                pd.DataFrame(llm_results).to_csv(out_path, mode="a", header=not os.path.exists(out_path), index=False)
                llm_results = []
                print(f"  Processed {i}/{len(futures)}")

    llm_df = pd.read_csv(out_path).drop_duplicates("recommendationid", keep="last")
    print(f"\nClassified {len(llm_df)} reviews. Parse errors: {(llm_df['reason'] == 'parse_error').sum()}, "
          f"API errors: {errors} (re-run the cell to retry both)")
    return llm_df


def batch_cost(llm_df):
    """Total cost of a batch, using the token counts Claude reported."""
    return (llm_df["input_tokens"] * INPUT_RATE + llm_df["output_tokens"] * OUTPUT_RATE
            + llm_df["cache_read_tokens"] * CACHE_READ_RATE + llm_df["cache_write_tokens"] * CACHE_WRITE_RATE).sum()

## Check the Prompt on the Dev Set

In [51]:
# Run Claude on the 312 pilot reviews (dev set) and compare with the pilot hand check
pilot_reviews = pd.read_csv("data/pilot.csv")
llm_dev = run_batch(pilot_reviews, "data/claude_labels_dev.csv")
print(llm_dev["reason"].value_counts().to_string())

# The 15 hand-checked pilot keyword hits (G = generative AI, E = enemy AI, N = no AI meaning)
hits_dev = pd.read_csv("data/pilot_hits_labeled.csv").merge(llm_dev, on="recommendationid")
hits_dev["claude_genai"] = hits_dev["reason"].isin(["GENAI", "BOTH"])
print("\n### Hand check vs. Claude ###")
print(pd.crosstab(hits_dev["label"], hits_dev["claude_genai"], rownames=["hand label"], colnames=["Claude: generative AI"]))

# Generative-AI flags the keywords missed (dev set, so reading them is allowed)
extra = pilot_reviews.merge(llm_dev, on="recommendationid")
extra = extra[extra["reason"].isin(["GENAI", "BOTH"]) & ~extra["recommendationid"].isin(hits_dev["recommendationid"])]
print(f"\n### Claude-only generative-AI flags: {len(extra)} ###")
for r in extra.itertuples():
    print(f"[{r.reason}, {r.confidence}] evidence: \"{r.evidence}\"\n   {r.review[:200]}\n")

print(f"Dev run: ${batch_cost(llm_dev):.2f} for {len(llm_dev)} reviews; "
      f"mean {llm_dev['response_time_s'].mean():.2f}s per review")

Classifying 312 reviews (0 already done)...
  Processed 50/312
  Processed 100/312
  Processed 150/312
  Processed 200/312
  Processed 250/312
  Processed 300/312
  Processed 312/312

Classified 312 reviews. Parse errors: 0, API errors: 0 (re-run the cell to retry both)
reason
QUALITY    272
OTHER       26
BOTH         8
SKIP         5
GENAI        1

### Hand check vs. Claude ###
Claude: generative AI  False  True 
hand label                         
E                          6      0
G                          0      8
N                          1      0

### Claude-only generative-AI flags: 1 ###
[BOTH, high] evidence: "I can feel a LOT more gAI in the game than just the 'steam capsule'... incredibly buggy (actual gameplay wise) and/or vague controls etc"
   If you believe in '1 buck in price = 1 hour gameplay', than I cannot recommend this title, as I finished it under 10 hrs (including starting over), and bought it for 11,79. It's early access so I hope

Dev run: $0.56 for 312 re

## Dev Check: Observations
- **312 pilot reviews classified with 0 parse errors**, for $0.56 (about $1.79 per 1,000 reviews) at 1.6 seconds per review.
- **Claude matched the pilot hand check on all 15 keyword hits:** it flagged all 8 generative-AI complaints and none of the 6 enemy-AI or 1 generic-slang matches. The keyword search was right on only 8 of 15 (53%).
- **Claude found 1 generative-AI complaint the keywords missed**, written as "gAI." That brings the pilot's generative-AI count to 9 (2.9%).
- **Caveat:** these results are optimistic, since the labeling guide's examples were written from the pilot. Test-set performance is the real measure.
- **No prompt changes were needed.** The prompt is frozen at labeling guide v4 and committed before the full run.

## Run on All Reviews (Prompt Frozen)

In [56]:
# Prompt frozen after the dev check (committed before this run): classify all 2,638 cleaned reviews
reviews = pd.read_csv("data/reviews_clean.csv")
llm_df = run_batch(reviews, "data/claude_labels.csv")
assert set(llm_df["recommendationid"]) == set(reviews["recommendationid"]), "Every review should have a Claude label"
assert (llm_df["reason"] != "parse_error").all(), "Re-run this cell to retry parse errors"

print(llm_df["reason"].value_counts().to_string())
print(f"\nNot English according to Claude: {(llm_df['english'] == False).sum()}")
print(pd.crosstab(reviews.merge(llm_df, on="recommendationid")["group"],
                  reviews.merge(llm_df, on="recommendationid")["reason"]))
print(f"\nFull run: ${batch_cost(llm_df):.2f} for {len(llm_df)} reviews; mean {llm_df['response_time_s'].mean():.2f}s per review")

Classifying 1 reviews (2637 already done)...
  Processed 1/1

Classified 2638 reviews. Parse errors: 0, API errors: 0 (re-run the cell to retry both)
reason
QUALITY    2234
OTHER       237
BOTH         87
GENAI        65
SKIP         15

Not English according to Claude: 16
reason     BOTH  GENAI  OTHER  QUALITY  SKIP
group                                       
control      10      8    163     1247    13
disclosed    77     57     74      987     2

Full run: $4.87 for 2638 reviews; mean 1.64s per review


## Full Run: Observations
- **All 2,638 reviews labeled** for $4.87 (about $1.85 per 1,000) at 1.6 seconds per review, about 15 minutes with 5 requests in parallel. 4 replies initially failed to parse; after raising max_tokens to 400 and retrying, all were labeled.
- **Claude flagged a generative-AI complaint (GENAI or BOTH) in 134 of 1,197 disclosed reviews (11.2%) and 18 of 1,441 control reviews (1.2%).**
- **Quality complaints (QUALITY or BOTH) were similar in both groups:** 88.9% disclosed vs. 87.2% control.
- **Control reviews more often gave no reason (OTHER):** 11.3% vs. 6.2%.
- **15 reviews were labeled SKIP and 16 flagged as not English;** these are removed before analysis.
- **These are preliminary:** Claude's labels before evaluation against hand labels, as review-level percentages without game-level confidence intervals.

## Round 2 Labeling Sheet

In [57]:
# Round 2: reviews Claude flagged as generative AI that weren't in round 1 (labeled blind to Claude's label)
key = pd.read_csv("data/labeling_key.csv")
claude_ai = llm_df.loc[llm_df["reason"].isin(["GENAI", "BOTH"]), "recommendationid"]
round2 = reviews[reviews["recommendationid"].isin(claude_ai) & ~reviews["recommendationid"].isin(key["recommendationid"])]
round2 = round2.sample(frac=1, random_state=21).reset_index(drop=True)
round2["sheet_id"] = range(1001, 1001 + len(round2))
print(f"Round 2: {len(round2)} reviews Claude flagged that weren't in round 1")

for path, frame in {
    "data/labeling_key_round2.csv": round2[["sheet_id", "recommendationid"]],
    "data/labels_round2_neha.csv": round2[["sheet_id", "review"]].assign(reason="", unsure="", note=""),
}.items():
    if not os.path.exists(path):
        frame.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"Wrote {path} ({len(frame)} rows)")
    else:
        print(f"{path} already exists; not overwriting")

Round 2: 5 reviews Claude flagged that weren't in round 1
Wrote data/labeling_key_round2.csv (5 rows)
Wrote data/labels_round2_neha.csv (5 rows)


----

# Part 6: Evaluation
**Purpose:** Score Claude and the keyword baseline against the hand-labeled test set (rounds 1 and 2), check accuracy by confidence and by subgroup, and estimate cost.

**Input:** `data/labels_round1_final.csv`, `data/labels_round2_neha.csv`, `data/labeling_key.csv`, `data/labeling_key_round2.csv`, `data/keyword_flags.csv`, `data/claude_labels.csv`

## Build the Test Set

In [59]:
# Combine round 1 (final) and round 2 hand labels with keyword flags and Claude labels
AI = {"GENAI", "BOTH"}

r1 = (pd.read_csv("data/labels_round1_final.csv", dtype={"reason": str})[["sheet_id", "reason"]]
      .merge(pd.read_csv("data/labeling_key.csv")[["sheet_id", "recommendationid"]], on="sheet_id"))
r2 = (load_labels("data/labels_round2_neha.csv").reset_index()[["sheet_id", "reason"]]
      .merge(pd.read_csv("data/labeling_key_round2.csv"), on="sheet_id"))
hand = pd.concat([r1.assign(round=1), r2.assign(round=2)])[["recommendationid", "reason", "round"]]
hand = hand.rename(columns={"reason": "human"})
hand["human"] = hand["human"].str.strip().str.upper()
assert set(hand["human"]) <= LABELS, f"Unexpected hand labels: {set(hand['human']) - LABELS}"
assert hand["recommendationid"].is_unique

ev = (reviews[["recommendationid", "appid", "name", "group", "genre"]]
      .merge(pd.read_csv("data/keyword_flags.csv"), on="recommendationid")
      .merge(llm_df[["recommendationid", "reason", "confidence"]].rename(columns={"reason": "claude"}), on="recommendationid")
      .merge(hand, on="recommendationid", how="left"))
ev["labeled"] = ev["human"].notna() & (ev["human"] != "SKIP")
ev["human_ai"] = ev["human"].isin(AI)
ev["claude_ai"] = ev["claude"].isin(AI)
ev["kw"] = ev["keyword_flag"].astype(bool)

# Strata: A = keyword-flagged (all hand-labeled), B1 = only Claude flagged (all hand-labeled),
#         B0 = flagged by neither (a random sample hand-labeled)
ev["stratum"] = np.where(ev["kw"], "A", np.where(ev["claude_ai"], "B1", "B0"))
labeled_any = ev["human"].notna()
assert labeled_any[ev["stratum"] == "A"].all(), "Every keyword-flagged review should be hand-labeled"
assert labeled_any[ev["stratum"] == "B1"].all(), "Every Claude-only flag should be hand-labeled (round 2)"

summary = ev.groupby("stratum").agg(reviews=("recommendationid", "count"), hand_labeled=("labeled", "sum"),
                                    human_ai=("human_ai", "sum"))
print("### Test set by stratum ###")
print(summary.to_string())
print(f"\nHand-labeled: {ev['human'].notna().sum()} (excluding {(ev['human'] == 'SKIP').sum()} SKIP)")

### Test set by stratum ###
         reviews  hand_labeled  human_ai
stratum                                 
A            236           236       140
B0          2397            99         0
B1             5             5         4

Hand-labeled: 341 (excluding 1 SKIP)


## Precision, Recall, and F1 (Generative-AI Class)

In [60]:
def ai_metrics(e):
    """Precision, recall and F1 for the generative-AI class, with recall weighted back to all 2,638 reviews."""
    A, B1, B0 = (e[e["stratum"] == s] for s in ["A", "B1", "B0"])
    A_l, B1_l, B0_l = A[A["labeled"]], B1[B1["labeled"]], B0[B0["labeled"]]
    p0 = B0_l["human_ai"].mean() if len(B0_l) else 0.0                      # AI rate among reviews neither method flagged
    total_ai = A_l["human_ai"].sum() + B1_l["human_ai"].sum() + p0 * len(B0)  # estimated AI complaints in all reviews
    kw_tp = A_l["human_ai"].sum()
    cl_flag = pd.concat([A_l[A_l["claude_ai"]], B1_l])
    cl_tp = cl_flag["human_ai"].sum()
    out = {}
    for name, tp, flagged in [("Keywords", kw_tp, len(A_l)), ("Claude", cl_tp, len(cl_flag))]:
        p = tp / flagged if flagged else np.nan
        r = tp / total_ai if total_ai else np.nan
        out[name] = {"precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0.0}
    out["_total_ai"] = total_ai
    return out

def boot_metrics(e, n_boot=2000, seed=42):
    """95% CIs by resampling the hand-labeled reviews within each stratum."""
    rng = np.random.default_rng(seed)
    parts = {s: e[e["stratum"] == s] for s in ["A", "B1", "B0"]}
    draws = []
    for _ in range(n_boot):
        sample = []
        for s, part in parts.items():
            lab, unlab = part[part["labeled"]], part[~part["labeled"]]
            sample.append(lab.sample(len(lab), replace=True, random_state=rng.integers(1e9)))
            sample.append(unlab)
        draws.append(ai_metrics(pd.concat(sample)))
    rows = []
    point = ai_metrics(e)
    for method in ["Keywords", "Claude"]:
        for m in ["precision", "recall", "f1"]:
            vals = [d[method][m] for d in draws]
            rows.append({"method": method, "metric": m, "estimate": point[method][m],
                         "ci_low": np.nanpercentile(vals, 2.5), "ci_high": np.nanpercentile(vals, 97.5)})
    return pd.DataFrame(rows).round(3), point["_total_ai"]

metrics, total_ai = boot_metrics(ev)
print("### Generative-AI class: precision, recall, F1 (95% bootstrap CIs) ###")
print(metrics.to_string(index=False))
print(f"\nEstimated generative-AI complaints in all {len(ev):,} reviews: {total_ai:.0f}")

# Conservative check: if the true AI rate among unflagged reviews were at its 95% upper bound
B0_l = ev[(ev["stratum"] == "B0") & ev["labeled"]]
k, n = int(B0_l["human_ai"].sum()), len(B0_l)
upper = wilson_ci(k, n)[1]
print(f"Unflagged sample: {k} of {n} were AI complaints (95% upper bound {upper:.1%}); "
      f"at that bound, about {upper * (ev['stratum'] == 'B0').sum():.0f} AI complaints would be missed by both methods")

### Generative-AI class: precision, recall, F1 (95% bootstrap CIs) ###
  method    metric  estimate  ci_low  ci_high
Keywords precision     0.593   0.530    0.653
Keywords    recall     0.972   0.962    0.986
Keywords        f1     0.737   0.685    0.782
  Claude precision     0.934   0.891    0.969
  Claude    recall     0.986   0.964    1.000
  Claude        f1     0.959   0.933    0.980

Estimated generative-AI complaints in all 2,638 reviews: 144
Unflagged sample: 0 of 99 were AI complaints (95% upper bound 3.7%); at that bound, about 90 AI complaints would be missed by both methods


## Precision, Recall, and F1: Observations
- **Claude is far more precise than the keyword baseline:** 0.93 (95% CI 0.89–0.97) vs. 0.59 (0.53–0.65). The intervals don't overlap. Claude flagged about 152 reviews with about 10 false alarms; the keywords flagged 236 with 96, mostly enemy-AI and generic "slop" mentions.
- **Claude's F1 is 0.96 (0.93–0.98) vs. 0.74 (0.69–0.78)** for keywords, a gap driven by precision.
- **Both methods have very high estimated recall** (Claude 0.99, keywords 0.97), finding an estimated 144 generative-AI complaints across all 2,638 reviews. Claude caught 4 complaints the keywords missed, all describing AI in other words ("computer voice," "TTS").
- **Recall is uncertain in one direction:** the random sample of 99 reviews neither method flagged had 0 AI complaints, but the true rate there could be up to 3.7% (95% upper bound). In that worst case about 90 complaints would be missed by both methods, and recall would fall to about 0.6 for both. The bootstrap intervals for recall understate this.

## Classification Report and Confusion Matrix

In [61]:
# Class-style report: Claude vs. hand labels on every hand-labeled review
lab = ev[ev["labeled"]]
classes = ["BOTH", "GENAI", "OTHER", "QUALITY", "SKIP"]
print(f"### Claude vs. hand labels on {len(lab)} hand-labeled reviews (weighted toward AI mentions) ###")
print(classification_report(lab["human"], lab["claude"], labels=classes, zero_division=0))
cm = confusion_matrix(lab["human"], lab["claude"], labels=classes)
print(pd.DataFrame(cm, index=[f"true:{c}" for c in classes], columns=[f"pred:{c}" for c in classes]))
print(f"\nCohen's kappa, Claude vs. hand labels (5 labels): {cohen_kappa_score(lab['human'], lab['claude']):.2f}")
print(f"Cohen's kappa, Claude vs. hand labels (AI vs. not): {cohen_kappa_score(lab['human_ai'], lab['claude_ai']):.2f}")
print(f"Cohen's kappa, keywords vs. hand labels (AI vs. not): {cohen_kappa_score(lab['human_ai'], lab['kw']):.2f}")

### Claude vs. hand labels on 340 hand-labeled reviews (weighted toward AI mentions) ###
              precision    recall  f1-score   support

        BOTH       0.75      0.88      0.81        74
       GENAI       0.88      0.81      0.84        70
       OTHER       0.78      0.64      0.70        11
     QUALITY       0.97      0.94      0.95       185
        SKIP       0.00      0.00      0.00         0

    accuracy                           0.89       340
   macro avg       0.67      0.65      0.66       340
weighted avg       0.89      0.89      0.89       340

              pred:BOTH  pred:GENAI  pred:OTHER  pred:QUALITY  pred:SKIP
true:BOTH            65           8           0             1          0
true:GENAI           12          57           0             1          0
true:OTHER            0           0           7             4          0
true:QUALITY         10           0           2           173          0
true:SKIP             0           0           0          

## Classification Report: Observations
- **Claude matched the final hand labels on 89% of 340 reviews (Cohen's κ = 0.82)**, higher than the two human labelers' agreement with each other (κ = 0.64). The final labels were settled after discussion, so this compares Claude against the best available answer key, not against one person.
- **On the key distinction, generative-AI complaint vs. not, Claude's κ is 0.93 vs. 0.44 for the keyword baseline.**
- **Most of Claude's errors (20 of 37) are GENAI vs. BOTH:** whether a review also contains a separate complaint. Human labelers found this distinction hardest too. Both labels count as AI complaints, so these errors don't affect the main comparison.
- **Claude's main false positives are 10 QUALITY reviews labeled BOTH**, likely mild or explicitly excused AI mentions, the hardest rule in the labeling guide.
- **The weakest class is OTHER (F1 0.70)**, based on only 11 reviews. QUALITY is the strongest (F1 0.95).
- **This test set is weighted toward AI mentions**, so per-class numbers describe these reviews, not all 2,638; the weighted estimates above are the main result.

## Threshold Back-Tests

Each governance rule in the memo is tested on our hand-labeled reviews, following the Class 18 back-test notebook. For each rule we ask two questions: how often would it fire, and does it catch the errors that matter?

An error here is a review where Claude and the hand label disagree on whether there is a generative-AI complaint. Mix-ups between GENAI and BOTH don't count, because they don't change the study's main number (the share of reviews with an AI complaint).

In [64]:
# Our results in the Class 18 format: one row per hand-labeled review
lab_all = ev[ev["labeled"]].copy()
genre_bucket = np.select(
    [lab_all["genre"].str.contains("shooter"), lab_all["genre"].str.contains("survival"), lab_all["genre"].str.contains("sim")],
    ["shooter", "survival", "sim"], default="other (RPG, tactics, detective)")

# Why: AI complaint vs. not is the error that changes our results
results = pd.DataFrame({
    "recommendationid": lab_all["recommendationid"],
    "true_label": np.where(lab_all["human_ai"], "genai_complaint", "no_genai_complaint"),
    "predicted_label": np.where(lab_all["claude_ai"], "genai_complaint", "no_genai_complaint"),
    "confidence": lab_all["confidence"],
    "source": genre_bucket,
    "group": lab_all["group"],
})
results["is_error"] = results["true_label"] != results["predicted_label"]
n_errors = results["is_error"].sum()

print(f"Items: {len(results)}   Errors: {n_errors} ({n_errors / len(results):.1%})")
print(f"(For reference, 5-label mismatches including GENAI vs. BOTH: {(lab_all['human'] != lab_all['claude']).sum()})")
print(results["confidence"].value_counts())

Items: 340   Errors: 12 (3.5%)
(For reference, 5-label mismatches including GENAI vs. BOTH: 38)
confidence
high      311
medium     28
low         1
Name: count, dtype: int64


**Results Table: Observations**
- Of the 340 hand-labeled reviews, Claude and the hand labels disagree on whether a review has a generative-AI complaint in only 12 (3.5%). These are the errors the back-tests check.
- The full 5-label mismatch count is 38, so 26 of the disagreements are mix-ups that don't change the main result (mostly GENAI vs. BOTH, or QUALITY vs. OTHER). The Section 3 rule that each class needs F1 ≥ 0.70 is what tests whether the GENAI vs. BOTH split can be reported.
- Claude marked almost every review high confidence (311 of 340, 91%). Only 1 review was low. A confidence rule can only work if confidence separates right answers from wrong ones, and Back-Test 1 checks whether it does.

### Back-Test 1: Confidence Threshold
Rule: **send a review to a human when Claude's confidence is low.**

In [65]:
flag_levels = ["low"]  # the rule being tested

flag = results["confidence"].isin(flag_levels)
flag_all = ev["confidence"].isin(flag_levels).mean()  # Why: workload is set by all reviews, not just labeled ones
caught = (flag & results["is_error"]).sum()
missed = n_errors - caught

print(f"Rule: human review when confidence is {flag_levels}")
print(f"Flags:   {flag.sum()}/{len(results)} labeled reviews ({flag.mean():.1%}); {flag_all:.1%} of all {len(ev):,} reviews")
print(f"Catches: {caught}/{n_errors} errors ({caught / n_errors:.0%})")
print(f"Misses:  {missed} errors the rule never sees")

if flag_all > 0.30:
    print("WARNING: Flags nearly a third or more of volume. Can your review team")
    print("  handle that at production scale? If not, tighten the threshold.")
elif flag_all < 0.02:
    print("WARNING: Almost nothing gets flagged. The rule may be decorative.")
if caught / n_errors < 0.50:
    print("WARNING: The rule misses most errors. Add a second safeguard, such as")
    print("  a random audit sample of the items it doesn't flag.")

print("Where the errors are:")
print(pd.crosstab(results["confidence"], results["is_error"], margins=True))

Rule: human review when confidence is ['low']
Flags:   1/340 labeled reviews (0.3%); 0.3% of all 2,638 reviews
Catches: 0/12 errors (0%)
Misses:  12 errors the rule never sees
  a random audit sample of the items it doesn't flag.
Where the errors are:
is_error    False  True  All
confidence                  
high          301    10  311
low             1     0    1
medium         26     2   28
All           328    12  340


**Back-Test 1: Observations**
- The low-confidence rule is essentially decorative. It flags 1 of 340 labeled reviews (0.3% of all 2,638) and caught 0 of the 12 errors.
- Confidence barely separates right answers from wrong ones. High-confidence reviews were wrong 3.2% of the time (10 of 311) and medium ones 7.1% (2 of 28). Most errors (10 of 12) are high-confidence, so no confidence cutoff can catch them.
- The AI-drafted rule ("human review when confidence is low") fails its back-test. Claude's self-reported confidence can't be the main safeguard. The human-review rule has to rest on the random audit of unflagged reviews, with confidence at most a supporting check (tested on low and medium in Back-Test 3).

### Back-Test 2: Subgroup Error Rate
Rule: **alert when any genre's or game group's error rate is above 15%, counting only subgroups with at least 30 reviews.**

In [66]:
threshold_error_rate = 0.15  # the rule being tested
min_n = 30                   # minimum reviews before a subgroup's rate counts

for col in ["source", "group"]:
    by_sub = results.groupby(col).agg(n=("is_error", "size"), errors=("is_error", "sum"))
    by_sub["error_rate"] = (by_sub["errors"] / by_sub["n"]).round(3)
    by_sub["would_trigger"] = by_sub["error_rate"] > threshold_error_rate
    by_sub["enough_data"] = by_sub["n"] >= min_n

    print(f"Rule: alert when {'genre' if col == 'source' else 'game group'} error rate > {threshold_error_rate:.0%}\n")
    print(by_sub.to_string())
    triggered = by_sub.index[by_sub["would_trigger"]].tolist()
    if not triggered:
        print("No subgroup crosses this threshold.")
    else:
        print(f"Would trigger for: {triggered}")
        small = by_sub.index[by_sub["would_trigger"] & ~by_sub["enough_data"]].tolist()
        if small:
            print(f"WARNING: {small} have fewer than {min_n} items. One or two errors")
            print("  decide this alert. Set a minimum sample size in your rule.")
    print()

# Check for uneven errors between groups
rates = results.groupby("group")["is_error"].mean()
print(f"Error-rate ratio (higher group / lower group): {rates.max() / rates.min():.2f}")
for t in [1.25, 1.5, 2.0]:
    print(f"  ratio > {t}: {'would trigger' if rates.max() / rates.min() > t else 'passes'}")
# Why: uneven errors between groups could bias the disclosed vs. control comparison

Rule: alert when genre error rate > 15%

                                   n  errors  error_rate  would_trigger  enough_data
source                                                                              
other (RPG, tactics, detective)   23       0       0.000          False        False
shooter                           55       2       0.036          False         True
sim                              206       9       0.044          False         True
survival                          56       1       0.018          False         True
No subgroup crosses this threshold.

Rule: alert when game group error rate > 15%

             n  errors  error_rate  would_trigger  enough_data
group                                                         
control    120       5       0.042          False         True
disclosed  220       7       0.032          False         True
No subgroup crosses this threshold.

Error-rate ratio (higher group / lower group): 1.31
  ratio > 1.25: would tri

**Back-Test 2: Observations**
- No genre comes close to the 15% alert. The highest is sim at 4.4% (9 of 206). Shooters, where we expected enemy "AI" to cause the most mistakes, are at 3.6% (2 of 55). The guide's enemy-AI rules appear to have worked.
- "Other" (RPG, tactics, detective) has only 23 reviews, under the 30-review minimum, so its 0% doesn't count. The minimum sample rule worked as intended.
- At 15%, the genre rule would never fire on our data, about 4× our overall error rate (3.5%). We keep it as a drift alarm for new data and lower it to 10% (about 3× the overall rate) so it would catch a real jump without firing on noise.
- Errors are slightly higher in control games (4.2%) than disclosed games (3.2%), a ratio of 1.31. It passes our 1.5 limit but would trigger the course example's 1.25.
- The ratio is fragile at these counts. Control has only 5 errors: one fewer gives a ratio of about 1.05, one more about 1.56. The memo should say a ratio built on fewer than about 10 errors per group is a signal to re-check, not proof of bias.

### Back-Test 3: Threshold Sensitivity
Try a few values for each rule. If a small change swings the result a lot, the threshold is fragile, and the memo should say so.

In [67]:
print("Confidence rule:")
for levels in [["low"], ["low", "medium"]]:
    f = results["confidence"].isin(levels)
    c = (f & results["is_error"]).sum()
    print(f"  flag {str(levels):20s} flags {ev['confidence'].isin(levels).mean():5.1%} of all reviews   catches {c}/{n_errors} errors")

print("Error-rate rule (genre):")
by_genre = results.groupby("source")["is_error"].mean()
for t in [0.05, 0.10, 0.15]:
    print(f"  > {t:.0%}: {by_genre.index[by_genre > t].tolist()}")

Confidence rule:
  flag ['low']              flags  0.3% of all reviews   catches 0/12 errors
  flag ['low', 'medium']    flags  7.1% of all reviews   catches 2/12 errors
Error-rate rule (genre):
  > 5%: []
  > 10%: []
  > 15%: []


**Back-Test 3: Observations**
- Widening the confidence rule to low or medium flags 7.1% of all reviews but still catches only 2 of 12 errors. Neither setting makes confidence a useful safeguard, because 10 of the 12 errors are high-confidence.
- We keep "low or medium" as a light supporting check. At 7.1% of volume a reviewer can keep up, and medium reviews are wrong about twice as often (7.1% vs. 3.2%). The random audit stays the main safeguard.
- No genre crosses 5%, 10%, or 15%, so the genre rule gives the same answer at every threshold we tried. It isn't fragile, but it hasn't been tested against a real problem. It stays as a drift alarm for new data, at 10%.

### Back-Test 4: Cost Cap
Rule: **Pause and review the setup if a run costs more than $3.00 per 1,000 reviews.**

In [68]:
COST_CAP_PER_1000 = 3.00  # the rule being tested
cost_per_1000 = batch_cost(llm_df) / len(llm_df) * 1000

print(f"Rule: stop and review if cost exceeds ${COST_CAP_PER_1000:.2f} per 1,000 reviews")
print(f"Actual: ${cost_per_1000:.2f} per 1,000 ({cost_per_1000 / COST_CAP_PER_1000:.0%} of the cap)")
for cap in [2.00, 3.00, 5.00]:
    print(f"  cap ${cap:.2f}: {'would trigger' if cost_per_1000 > cap else 'passes'}")

Rule: stop and review if cost exceeds $3.00 per 1,000 reviews
Actual: $1.84 per 1,000 (61% of the cap)
  cap $2.00: passes
  cap $3.00: passes
  cap $5.00: passes


**Back-Test 4: Observations**
- The full run cost $1.84 per 1,000 reviews (about $4.85 for all 2,638), 61% of the $3.00 cap.
- A $2.00 cap would be too tight. We are already at 92% of it, so a batch of longer reviews could trip it with nothing wrong.
- A $5.00 cap would be too loose to catch the most likely failure, prompt caching breaking. The labeling guide goes out with every review. Cached, it costs very little. Uncached, the guide alone would cost roughly $7 per 1,000 reviews. The $3.00 cap leaves room for normal variation but would stop a run where caching failed.
- Cost is low compared with hand labeling, so this rule protects against a broken setup, not a budget overrun.

### Back-Test 5: Random Audit
Rule: **Each run, a person checks 100 random reviews Claude didn't flag; stop if more than 3 are missed generative-AI complaints.**

In [69]:
# Claude-unflagged reviews are stratum A (keyword-flagged, labeled in full) plus B0 (unflagged by both, sampled)
assert set(ev["stratum"]) == {"A", "B1", "B0"}
missed = lab_all["human_ai"] & ~lab_all["claude_ai"]
in_A, in_B0 = lab_all["stratum"] == "A", lab_all["stratum"] == "B0"

# Why: weight the B0 sample back up to all B0 reviews, as in the recall estimate
n_unflagged = (~ev["claude_ai"]).sum()
est_missed = missed[in_A].sum() + missed[in_B0].mean() * (ev["stratum"] == "B0").sum()
per_100 = est_missed / n_unflagged * 100

print(f"Missed AI complaints: {missed[in_A].sum()} in stratum A, {missed[in_B0].sum()} of {in_B0.sum()} in the B0 sample")
print(f"Estimated missed in all {n_unflagged:,} unflagged reviews: {est_missed:.1f}")
print(f"Expected per 100-review audit: {per_100:.2f}")
for cap in [1, 3, 5]:
    print(f"  stop if > {cap}: {'would trigger' if per_100 > cap else 'passes'}")

Missed AI complaints: 2 in stratum A, 0 of 99 in the B0 sample
Estimated missed in all 2,486 unflagged reviews: 2.0
Expected per 100-review audit: 0.08
  stop if > 1: passes
  stop if > 3: passes
  stop if > 5: passes


**Back-Test 5: Observations**
- Claude rarely misses an AI complaint. Only 2 of the 12 errors are misses, both in stratum A, and 0 of the 99 sampled B0 reviews were missed. That works out to about 2 missed complaints across all 2,486 unflagged reviews, or 0.08 per 100-review audit.
- The other 10 errors are false alarms, where Claude labeled an AI complaint the hand label didn't. Claude's mistakes lean toward over-flagging, so uncorrected labels would slightly overstate generative-AI complaints.
- This changes the human-review rule. Auditing unflagged reviews would almost never find an error. A person checking every review Claude labels as an AI complaint (152 reviews, 5.8% of volume) would catch 10 of the 12 errors.
- The random audit of 100 unflagged reviews stays as a smaller check on missed complaints. With 0 of 99 missed, the 95% upper bound is about 3.7 per 100, so "stop if more than 3" is set right at the edge of what our data allows. Crossing it would mean recall has really dropped.

## Cost Analysis
Claude's cost per review compared with hand labeling, at our own labeling pace. The governed cost adds the human checks the memo requires: a person checks every review Claude labels as an AI complaint, plus a 100-review random audit per run.

In [70]:
total_cost = batch_cost(llm_df)
cost_per_text = total_cost / len(llm_df)
mean_time = llm_df["response_time_s"].mean()
print(f"Full run of {len(llm_df):,} reviews:")
print(f"  Input tokens: {llm_df['input_tokens'].sum():,}  Cached tokens: {llm_df['cache_read_tokens'].sum():,}  Output tokens: {llm_df['output_tokens'].sum():,}")
print(f"  Total cost: ${total_cost:.2f}   Cost per review: ${cost_per_text:.5f}   Mean time: {mean_time:.2f}s")

print("\n### Cost Projections ###")
for n in [1_000, 10_000, 100_000]:
    print(f"  {n:>7,} reviews:  ${cost_per_text * n:>8.2f}  ({mean_time * n / 60 / 5:>5.0f} min with 5 parallel requests)")

HUMAN_RATE = 25.00     # $ per hour, as in class
TEXTS_PER_HOUR = 57    # our hand-labeling pace: about 100 reviews in 1.5-2 hours
FLAG_SHARE = ev["claude_ai"].mean()  # share of reviews Claude labels as AI complaints, all checked by a person
AUDIT_N = 100          # random unflagged reviews audited per run

print(f"\n### Human vs. LLM Cost Comparison (human pace {TEXTS_PER_HOUR}/hour) ###")
for n in [1_000, 10_000]:
    human_cost = n / TEXTS_PER_HOUR * HUMAN_RATE
    llm_cost = cost_per_text * n
    # Why: the memo's human checks are part of the real cost of using Claude
    governed_cost = llm_cost + (n * FLAG_SHARE + AUDIT_N) / TEXTS_PER_HOUR * HUMAN_RATE
    print(f"  {n:>7,} reviews:  Human=${human_cost:>9.2f}  LLM=${llm_cost:>6.2f}  "
          f"LLM + human checks=${governed_cost:>8.2f}  ({human_cost / governed_cost:.0f}x cheaper with checks)")

print("\nHuman cost range at our slowest and fastest pace, 1,000 reviews:")
for pace in [50, 67]:
    print(f"  {pace}/hour: ${1_000 / pace * HUMAN_RATE:.2f}")

Full run of 2,638 reviews:
  Input tokens: 399,349  Cached tokens: 6,682,554  Output tokens: 107,665
  Total cost: $4.87   Cost per review: $0.00184   Mean time: 1.64s

### Cost Projections ###
    1,000 reviews:  $    1.84  (    5 min with 5 parallel requests)
   10,000 reviews:  $   18.44  (   55 min with 5 parallel requests)
  100,000 reviews:  $  184.43  (  545 min with 5 parallel requests)

### Human vs. LLM Cost Comparison (human pace 57/hour) ###
    1,000 reviews:  Human=$   438.60  LLM=$  1.84  LLM + human checks=$   70.98  (6x cheaper with checks)
   10,000 reviews:  Human=$  4385.96  LLM=$ 18.44  LLM + human checks=$  315.02  (14x cheaper with checks)

Human cost range at our slowest and fastest pace, 1,000 reviews:
  50/hour: $500.00
  67/hour: $373.13


**Cost Analysis: Observations**
- Labeling all 2,638 reviews cost $4.87 ($1.84 per 1,000) and took about 15 minutes with 5 parallel requests. Hand labeling at our pace (57 per hour) would take about 46 hours and cost about $1,150 at $25/hour.
- Prompt caching made the run affordable. 6.7 million of the input tokens were the cached labeling guide. Sent uncached, they would have cost about $20 instead of $2, for a total of roughly $23 ($8.70 per 1,000). This confirms the $3.00 cost cap would catch a caching failure.
- Claude alone looks about 240x cheaper than hand labeling, but that isn't the honest comparison. With the human checks the memo requires (every AI flag plus a 100-review audit), 1,000 reviews cost $70.98, 6x cheaper than labeling by hand. At 10,000 reviews it's 14x cheaper, because the audit is a fixed 100 reviews per run.
- Almost all of the governed cost is the human checks, not Claude: about $69 of the $71 per 1,000. The cost of this workflow is set by how much human review the governance rules require, not by the API.
- Even at our fastest pace (67 per hour, $373 per 1,000 reviews), hand labeling costs over 5x the governed workflow.